### Realizamos los imports que requeriremos para el analisis

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set de opciones de pandas.
pd.set_option('display.max_columns', None)

# 02 Limpieza de datos 

### Objetivo

En esta fase realizaremos la limpieza y preparación del dataset de Spotify a partir de los problemas identificados durante el análisis exploratorio.

El objetivo es obtener conjuntos de datos fiables y estructurados que permitan desarrollar posteriormente los modelos de Machine Learning, el SOM y la aplicación gráfica con Tkinter.

In [46]:
df = pd.read_csv('../Datos/datosSpotify.csv', index_col=0)

# Crear copia de trabajo
df_limpio = df.copy()

print("Dataset original:", df.shape)
display(df_limpio.head())

Dataset original: (114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 2.1 Eliminación de duplicados exactos

Durante el análisis exploratorio identificamos 450 filas completamente duplicadas.

- Registros iniciales: 114.000.
- Duplicados eliminados: 450.
- Registros restantes: 113.550.
- Canciones únicas (`track_id`): 89.741.

In [47]:
# Guardar número de filas inicial
filas_antes = len(df_limpio)

# Eliminar duplicados exactos
df_limpio = df_limpio.drop_duplicates().copy()

# Calcular resultados
filas_despues = len(df_limpio)
duplicados_eliminados = filas_antes - filas_despues

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Duplicados eliminados:", duplicados_eliminados)

print("\nCanciones únicas:", df_limpio['track_id'].nunique())


Filas antes: 114000
Filas después: 113550
Duplicados eliminados: 450

Canciones únicas: 89741


## 2.2 Eliminacion de valores nulos

Durante la inspección se identificó una única fila con valores nulos en las variables `artists`, `album_name` y `track_name`.

Como los tres valores ausentes pertenecen al mismo registro y dificultan la identificación de la canción, se ha decidido eliminar dicha fila.

In [48]:

# Registrar filas antes de la eliminación
filas_antes = len(df_limpio)

# Eliminar registros con nulos en las variables identificativas
df_limpio = df_limpio.dropna(
    subset=['artists', 'album_name', 'track_name']
).copy()

# Comprobar resultados
print("Filas eliminadas:", filas_antes - len(df_limpio))
print("Filas restantes:", len(df_limpio))

print("\nValores nulos restantes:")
print(df_limpio.isnull().sum())


Filas eliminadas: 1
Filas restantes: 113549

Valores nulos restantes:
track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


## 2.3 Valores anómalos y variables no relevantes

Según lo identificado en el análisis exploratorio (apartado 1.5):

- Se eliminan las **157 canciones con `tempo = 0`**. Corresponden en su mayoría a pistas
  de ruido blanco o ambientales (género *sleep*) en las que Spotify no detecta ritmo.
- Se elimina la variable **`time_signature`**, ya que casi todas las canciones están en 4/4,
  contiene valores no válidos (0 y 1) y no muestra relación con la popularidad.
- Las duraciones extremas se **mantienen**, ya que no son errores sino valores atípicos reales.

In [49]:
filas_antes = len(df_limpio)
canciones_antes = df_limpio['track_id'].nunique()

# Eliminar canciones con tempo 0
df_limpio = df_limpio[df_limpio['tempo'] > 0].copy()

print("Filas eliminadas por tempo = 0:", filas_antes - len(df_limpio))
print("Canciones eliminadas:", canciones_antes - df_limpio['track_id'].nunique())

# Eliminar la variable time_signature
df_limpio = df_limpio.drop(columns=['time_signature'])

print("\nFilas restantes:", len(df_limpio))
print("Canciones únicas:", df_limpio['track_id'].nunique())
print("Columnas restantes:", df_limpio.shape[1])

Filas eliminadas por tempo = 0: 157
Canciones eliminadas: 157

Filas restantes: 113392
Canciones únicas: 89583
Columnas restantes: 19



## 2.4 Tratamiento de las inconsistencias de popularidad

Durante el análisis exploratorio se detectaron diferentes valores de popularidad para un mismo `track_id`.

Se aplicarán los siguientes criterios:

- Si todos los registros tienen la misma popularidad, se conservará ese valor.
- Si existen valores cero junto con valores positivos, se excluirán los ceros para calcular el valor representativo.
- Si existen diferentes valores positivos con una diferencia máxima de cinco puntos, se utilizará la mediana.
- Si la diferencia entre los valores válidos supera los cinco puntos, se eliminará la canción del dataset final.

La mediana se utilizará como medida representativa para evitar favorecer sistemáticamente los valores más altos o más bajos.



In [50]:
# Estadísticas de popularidad de cada canción
resumen_pop = (
    df_limpio
    .groupby('track_id')['popularity']
    .agg(
        apariciones='size',
        valores_distintos='nunique',
        popularidad_min_original='min',
        popularidad_max_original='max',
        mediana_original='median'
    )
)

# Estadísticas considerando solamente valores positivos
estadisticas_positivas = (
    df_limpio[df_limpio['popularity'] > 0]
    .groupby('track_id')['popularity']
    .agg(
        cantidad_positiva='size',
        popularidad_min_positiva='min',
        popularidad_max_positiva='max',
        mediana_positiva='median'
    )
)

# Incorporar estadísticas positivas
resumen_pop = resumen_pop.join(estadisticas_positivas)

# Identificar canciones con ceros y valores positivos
mezcla_ceros = (
    (resumen_pop['popularidad_min_original'] == 0) &
    (resumen_pop['popularidad_max_original'] > 0)
)

# Calcular cuántos ceros se excluyen
resumen_pop['ceros_excluidos'] = np.where(
    mezcla_ceros,
    resumen_pop['apariciones'] - resumen_pop['cantidad_positiva'],
    0
)

# Calcular mínimo, máximo y mediana válidos
resumen_pop['min_valido'] = np.where(
    mezcla_ceros,
    resumen_pop['popularidad_min_positiva'],
    resumen_pop['popularidad_min_original']
)

resumen_pop['max_valido'] = np.where(
    mezcla_ceros,
    resumen_pop['popularidad_max_positiva'],
    resumen_pop['popularidad_max_original']
)

resumen_pop['popularidad_mediana'] = np.where(
    mezcla_ceros,
    resumen_pop['mediana_positiva'],
    resumen_pop['mediana_original']
)

# Diferencia entre valores válidos
resumen_pop['diferencia_valida'] = (
    resumen_pop['max_valido'] - resumen_pop['min_valido']
)

# Identificar anomalías
resumen_pop['excluir_popularidad'] = (
    resumen_pop['diferencia_valida'] > 5
)

# Popularidad definitiva: NaN para canciones con discrepancias excesivas
resumen_pop['popularidad_consolidada'] = np.where(
    resumen_pop['excluir_popularidad'],
    np.nan,
    resumen_pop['popularidad_mediana']
)


In [51]:
print("Canciones únicas:", len(resumen_pop))

print(
    "Canciones con ceros y valores positivos:",
    mezcla_ceros.sum()
)

print(
    "Registros cero excluidos en la consolidación:",
    int(resumen_pop['ceros_excluidos'].sum())
)

print(
    "Canciones excluidas del análisis de popularidad:",
    int(resumen_pop['excluir_popularidad'].sum())
)

print(
    "Canciones con popularidad consolidada:",
    int(resumen_pop['popularidad_consolidada'].notna().sum())
)

# Mostrar posibles anomalías
display(
    resumen_pop[resumen_pop['excluir_popularidad']]
    .sort_values('diferencia_valida', ascending=False)
)


Canciones únicas: 89583
Canciones con ceros y valores positivos: 105
Registros cero excluidos en la consolidación: 185
Canciones excluidas del análisis de popularidad: 2
Canciones con popularidad consolidada: 89581


,apariciones,valores_distintos,popularidad_min_original,popularidad_max_original,mediana_original,cantidad_positiva,popularidad_min_positiva,popularidad_max_positiva,mediana_positiva,ceros_excluidos,min_valido,max_valido,popularidad_mediana,diferencia_valida,excluir_popularidad,popularidad_consolidada
track_id,,,,,,,,,,,,,,,,
6mPet122yXpp6X8iTiNVo9,6,2,4,16,4.0,6.0,4.0,16.0,4.0,0.0,4.0,16.0,4.0,12.0,True,NaN
2s3Hr38fRfDRj9JWVP494L,2,2,7,14,10.5,2.0,7.0,14.0,10.5,0.0,7.0,14.0,10.5,7.0,True,NaN


## 2.5 Consolidación de canciones y géneros

El dataset contiene canciones que aparecen en varios registros debido a su asociación con diferentes géneros musicales.

Para evitar duplicidades, se construirá un único dataset con una fila por `track_id`.

El tratamiento consistirá en:

- Agrupar los diferentes géneros asociados a cada canción en una única variable (`track_genres`).
- Conservar una única fila representativa por canción.
- Incorporar la popularidad consolidada mediante la mediana, siguiendo los criterios establecidos anteriormente.
- Eliminar del dataset las dos canciones con discrepancias de popularidad superiores a cinco puntos,
  ya que no es posible determinar cuál es su valor correcto.

De esta manera, cada canción estará representada una sola vez, sin perder sus asociaciones con los distintos géneros musicales.

Esta estructura será la base de los posteriores análisis exploratorios, modelos de Machine Learning y SOM.


In [52]:
# Comprobar que los metadatos y las características musicales
# son iguales en todos los registros de una misma canción.
columnas_comparar = [
    col for col in df_limpio.columns
    if col not in ['track_genre', 'popularity']
]

variaciones_finales = (
    df_limpio
    .groupby('track_id')[columnas_comparar]
    .nunique(dropna=False)
)

resumen_consistencia = (
    (variaciones_finales > 1)
    .sum()
    .sort_values(ascending=False)
)

diferencias = resumen_consistencia[resumen_consistencia > 0]

if not diferencias.empty:
    display(diferencias.to_frame(name='Canciones con diferencias'))
    raise ValueError(
        "No se puede consolidar todavía: hay metadatos o características "
        "que cambian entre registros del mismo track_id."
    )

print("Correcto: los metadatos y las características musicales son consistentes.")


Correcto: los metadatos y las características musicales son consistentes.


In [53]:
# Identificar las dos canciones que vamos a excluir
ids_excluidos = resumen_pop[
    resumen_pop['excluir_popularidad']
].index

print("Canciones que se excluirán:", len(ids_excluidos))

# Excluir completamente estas canciones del dataset de trabajo
df_para_final = df_limpio[
    ~df_limpio['track_id'].isin(ids_excluidos)
].copy()

# Agrupar los géneros únicos de cada canción
generos_por_cancion = (
    df_para_final
    .groupby('track_id')['track_genre']
    .agg(lambda x: '; '.join(sorted(x.unique())))
)

# Seleccionar las columnas descriptivas y musicales
columnas_cancion = [
    col for col in df_para_final.columns
    if col not in ['track_genre', 'popularity']
]

# Conservar una única fila por canción
df_final = (
    df_para_final[columnas_cancion]
    .drop_duplicates(subset='track_id')
    .copy()
)

# Incorporar los géneros agrupados
df_final['track_genres'] = df_final['track_id'].map(
    generos_por_cancion
)

# Incorporar la popularidad consolidada
df_final['popularity'] = df_final['track_id'].map(
    resumen_pop['popularidad_consolidada']
)

# Mantener un orden de columnas organizado
columnas_finales = [
    'track_genres' if col == 'track_genre' else col
    for col in df_limpio.columns
    if col not in ['popularity']
]

columnas_finales.append('popularity')

df_final = df_final[columnas_finales]

print("Dimensiones del dataset final:", df_final.shape)
display(df_final.head())


Canciones que se excluirán: 2
Dimensiones del dataset final: (89581, 19)


,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genres,popularity
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,acoustic; j-pop; singer-songwriter; songwriter,73.0
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,acoustic; chill,55.0
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,acoustic,57.0
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,acoustic,71.0
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,acoustic,82.0



## 2.6 Validación y exportación del dataset final

Antes de guardar el archivo, comprobaremos que:

- Existe una única fila por `track_id`.
- El número de canciones coincide con las canciones tras la limpieza menos las dos excluidas.
- Todas las canciones tienen al menos un género y un valor de popularidad.
- Se conservan todas las relaciones únicas entre canciones y géneros.
- No quedan canciones con `tempo = 0` ni la variable `time_signature`.

Una vez superadas estas comprobaciones, guardaremos el resultado en `Datos/Procesados/spotify_limpio.csv`.

Este será el único dataset consolidado que utilizaremos en las siguientes fases del proyecto.


In [54]:
# Validaciones del dataset final
relaciones_esperadas = (
    df_para_final[['track_id', 'track_genre']]
    .drop_duplicates()
    .shape[0]
)

relaciones_finales = (
    df_final['track_genres']
    .str.split('; ')
    .str.len()
    .sum()
)

canciones_esperadas = df_limpio['track_id'].nunique() - len(ids_excluidos)

print("¿Todos los track_id son únicos?", df_final['track_id'].is_unique)
print("Canciones tras la limpieza:", df_limpio['track_id'].nunique())
print("Canciones excluidas por popularidad:", len(ids_excluidos))
print("Canciones en dataset final:", len(df_final))
print("Géneros diferentes:", df_para_final['track_genre'].nunique())
print("Canciones sin géneros:", df_final['track_genres'].isna().sum())
print("Canciones sin popularidad:", df_final['popularity'].isna().sum())
print("Relaciones originales conservadas:", int(relaciones_finales))

# Comprobaciones automáticas
assert df_final['track_id'].is_unique
assert len(df_final) == canciones_esperadas
assert df_final['popularity'].notna().all()
assert df_final['track_genres'].notna().all()
assert int(relaciones_finales) == relaciones_esperadas
assert (df_final['tempo'] > 0).all()
assert 'time_signature' not in df_final.columns

print("\nTodas las comprobaciones se han superado.")

# Guardar el dataset definitivo
carpeta_salida = Path('../Datos/Procesados')
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_salida / 'spotify_limpio.csv'
df_final.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"\nArchivo guardado en: {ruta_salida}")
print("Registros guardados:", len(df_final))
print("Columnas guardadas:", df_final.shape[1])

¿Todos los track_id son únicos? True
Canciones tras la limpieza: 89583
Canciones excluidas por popularidad: 2
Canciones en dataset final: 89581
Géneros diferentes: 114
Canciones sin géneros: 0
Canciones sin popularidad: 0
Relaciones originales conservadas: 113384

Todas las comprobaciones se han superado.

Archivo guardado en: ..\Datos\Procesados\spotify_limpio.csv
Registros guardados: 89581
Columnas guardadas: 19
